# Feature Engineering & Predictor Matrix Preparation

**Project:** OneHealth-ML  
**Purpose:**  
- Load the labeled master dataset  
- Separate administrative, survey‑design, label, and predictor variables  
- Create derived features (HOMA‑IR, NLR, WHR, etc.)  
- Apply the predictor‑leakage policy (exclude label‑defining variables per disease)  
- Impute missing values and encode categorical variables  
- Build final feature matrix **X** and label vectors **y** for all 7 diseases  
- Save processed data to `data/processed/`

## Imports

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import json

sys.path.append("..")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## Paths & Settings

In [2]:
PROCESSED_DIR = "../data/processed"
MASTER_PATH = os.path.join(PROCESSED_DIR, "labeled_master.csv")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Load Labeled Master Dataset

In [3]:
df = pd.read_csv(MASTER_PATH)
print(f"Loaded master dataset: {df.shape}")
print(f"Columns: {len(df.columns)}")

Loaded master dataset: (2541, 454)
Columns: 454


## 1. Identify Variable Categories
We manually classify every column into:  
- **Admin/ID**: SEQN, weights, PSU, strata, exam status flags  
- **Survey metadata**: not used in ML but kept for reference  
- **Disease labels**: columns starting with `label_`  
- **Raw biomarkers** that will be used for derived features (temporarily separated)  
- **Predictors**: all remaining clinical variables

In [4]:
# Administrative & survey design variables (never used as predictors)
ADMIN_VARS = [
    "SEQN", "WTMEC2YR", "SDMVPSU", "SDMVSTRA",  # survey weights/design
    "RIDSTATR", "BMDSTATS",                     # exam status flags
    "WTSAF2YR",                                  # fasting subsample weight
]

# Label columns (created in 02 notebook)
LABEL_COLS = [c for c in df.columns if c.startswith("label_")]

# Columns that are raw components for derived features (will be excluded from final X)
DERIVED_SOURCE_COLS = [
    "LBXLYPCT", "LBXNEPCT",  # NLR components
    "LBXIN", "LBXGLU",       # HOMA-IR components (insulin, glucose)
    "BMXHIP",                # WHR component (waist already in predictors)
]

# The remaining columns are predictors (excluding ID/labels/derived sources)
PREDICTOR_COLS = [
    c for c in df.columns
    if c not in ADMIN_VARS + LABEL_COLS + DERIVED_SOURCE_COLS
]

print(f"Administrative variables: {len(ADMIN_VARS)}")
print(f"Label columns: {len(LABEL_COLS)}")
print(f"Derived source columns (temporary): {len(DERIVED_SOURCE_COLS)}")
print(f"Predictor columns (before derived): {len(PREDICTOR_COLS)}")

Administrative variables: 7
Label columns: 7
Derived source columns (temporary): 5
Predictor columns (before derived): 435


## 2. Create Derived Features
Compute clinical ratios and indices that are standard in practice.

In [5]:
# --- Neutrophil-to-Lymphocyte Ratio (NLR) ---
df["NLR"] = df["LBXNEPCT"] / df["LBXLYPCT"].replace(0, np.nan)
df["NLR"] = df["NLR"].replace([np.inf, -np.inf], np.nan)

# --- HOMA-IR (Homeostatic Model Assessment of Insulin Resistance) ---
# HOMA-IR = (fasting insulin (µU/mL) * fasting glucose (mg/dL)) / 405
df["HOMA_IR"] = (df["LBXIN"] * df["LBXGLU"]) / 405

# --- Waist-to-Hip Ratio (WHR) ---
df["WHR"] = df["BMXWAIST"] / df["BMXHIP"].replace(0, np.nan)

# Add these derived features to the predictor list
DERIVED_COLS = ["NLR", "HOMA_IR", "WHR"]
PREDICTOR_COLS.extend(DERIVED_COLS)

print(f"Predictor columns (after derived): {len(PREDICTOR_COLS)}")

Predictor columns (after derived): 438


C:\Users\MiR\AppData\Local\Temp\ipykernel_13936\3129253048.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["NLR"] = df["LBXNEPCT"] / df["LBXLYPCT"].replace(0, np.nan)
C:\Users\MiR\AppData\Local\Temp\ipykernel_13936\3129253048.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["HOMA_IR"] = (df["LBXIN"] * df["LBXGLU"]) / 405
C:\Users\MiR\AppData\Local\Temp\ipykernel_13936\3129253048.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor

## 3. Predictor-Leakage Policy
For each disease, certain predictors must be **excluded** because they were used to define the label.  
We define an exclusion list per disease.

In [6]:
LEAKAGE_MAP = {
    "label_diabetes": [
        "LBXGH",      # HbA1c
        "LBXGLU",     # fasting glucose (also used in HOMA-IR, but we exclude the raw)
        "DIQ010",     # doctor diagnosed diabetes
        "HOMA_IR",    # derived from glucose & insulin, so exclude from diabetes model
    ],
    "label_hypertension": [
        "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4",
        "BPXPLS",      # pulse (if considered part of BP exam)
        "BPXPULS",
        "BPQ020",      # doctor diagnosis
        "mean_sbp",    # derived from BPs
        "mean_dbp",
    ],
    "label_ckd": [
        "LBXSCR",      # serum creatinine
        "eGFR",        # derived from creatinine
        "KIQ022",      # weak kidneys
        "KIQ025",      # dialysis
        "URDACT",      # urine ACR (label component)
    ],
    "label_dyslipidemia": [
        "LBXTC",       # total cholesterol
        "LBDHDD",      # HDL
        "LBXTR",       # triglycerides
    ],
    "label_cvd": [
        "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F",  # self-reported CVD
    ],
    "label_metabolic_syndrome": [
        "BMXWAIST",    # waist circumference
        "LBXTR",       # triglycerides
        "LBDHDD",      # HDL
        "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4",
        "mean_sbp", "mean_dbp",
        "LBXGLU",      # fasting glucose
        "WHR",         # contains waist
    ],
    "label_liver_disease": [
        "LBXSATSI",    # ALT
        "LBXSASSI",    # AST
        "LBXSGTSI",    # GGT
        "BMXBMI",      # BMI (part of label definition)
    ],
}

# Save leakage map for later use
with open(os.path.join(PROCESSED_DIR, "leakage_map.json"), "w") as f:
    json.dump(LEAKAGE_MAP, f, indent=2)

print("Leakage map saved.")

Leakage map saved.


## 4. Build Feature Matrix and Label Vectors
For each disease, we create a clean, imputed **X** and **y**.

In [7]:
from sklearn.model_selection import train_test_split

# We'll build separate X and y for each disease
feature_sets = {}
label_sets = {}

for label_col in LABEL_COLS:
    # Select rows where label is not NaN
    mask = df[label_col].notna()
    y = df.loc[mask, label_col].astype(int)
    
    # Select predictors, excluding leakage columns for this disease
    exclude = LEAKAGE_MAP.get(label_col, [])
    predictor_cols = [c for c in PREDICTOR_COLS if c not in exclude]
    
    X = df.loc[mask, predictor_cols].copy()
    
    feature_sets[label_col] = X
    label_sets[label_col] = y
    
    print(f"{label_col}: X.shape={X.shape}, y.shape={y.shape}, "
          f"positive={y.sum()}, negative={len(y)-y.sum()}")

label_diabetes: X.shape=(2520, 435), y.shape=(2520,), positive=583, negative=1937
label_hypertension: X.shape=(2541, 425), y.shape=(2541,), positive=1475, negative=1066
label_ckd: X.shape=(2541, 433), y.shape=(2541,), positive=247, negative=2294
label_dyslipidemia: X.shape=(2399, 435), y.shape=(2399,), positive=1388, negative=1011
label_cvd: X.shape=(2541, 433), y.shape=(2541,), positive=320, negative=2221
label_metabolic_syndrome: X.shape=(2192, 424), y.shape=(2192,), positive=850, negative=1342
label_liver_disease: X.shape=(2535, 434), y.shape=(2535,), positive=361, negative=2174


## 5. Preprocessing Pipeline
Impute missing values (median for numeric, mode for categorical) and scale.  
We treat all variables as numeric for simplicity; categorical variables have already been encoded with ordinal/label encoding where appropriate.

In [8]:
# Diagnostic: find which disease has object columns
for label_col in LABEL_COLS:
    X = feature_sets[label_col]
    obj_cols = [c for c in X.columns if X[c].dtype == object]
    if obj_cols:
        print(f"\n{label_col} has object columns:")
        for col in obj_cols:
            vals = X[col].dropna().unique()
            print(f"  {col}: values={vals[:10]}")
    else:
        print(f"{label_col}: all numeric ✓")

label_diabetes: all numeric ✓
label_hypertension: all numeric ✓
label_ckd: all numeric ✓
label_dyslipidemia: all numeric ✓
label_cvd: all numeric ✓
label_metabolic_syndrome: all numeric ✓
label_liver_disease: all numeric ✓


In [9]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

processed_X = {}

for label_col in LABEL_COLS:
    X = feature_sets[label_col].copy()
    
    # ---------- 1. Clean any bytes or mixed-type entries ----------
    for col in X.columns:
        # Decode bytes to string if necessary (preserves object dtype)
        if X[col].dtype == object:
            X[col] = X[col].apply(
                lambda x: x.decode('utf-8') if isinstance(x, bytes) else x
            )
        # Now force to numeric; any unparseable strings become NaN
        X[col] = pd.to_numeric(X[col], errors='coerce')
    
    # ---------- 2. Verify that every column is now numeric ----------
    still_object = [c for c in X.columns if X[c].dtype == object]
    if still_object:
        print(f"❌ {label_col}: The following columns are STILL non‑numeric after conversion:")
        for c in still_object:
            print(f"   {c}: unique values = {X[c].dropna().unique()[:10]}")
        # Stop processing this disease – we need to investigate
        continue
    
    # ---------- 3. Drop columns that became entirely NaN ----------
    before = X.shape[1]
    X = X.dropna(axis=1, how='all')
    after = X.shape[1]
    if after < before:
        print(f"⚠ {label_col}: dropped {before - after} column(s) that became all‑NaN")
    
    # ---------- 4. Pipeline ----------
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    
    X_processed = pipeline.fit_transform(X)
    processed_X[label_col] = X_processed
    print(f"{label_col}: processed shape {X_processed.shape}")

print("\nDone. Keys in processed_X:", list(processed_X.keys()))

⚠ label_diabetes: dropped 43 column(s) that became all‑NaN
label_diabetes: processed shape (2520, 392)
⚠ label_hypertension: dropped 42 column(s) that became all‑NaN
label_hypertension: processed shape (2541, 383)
⚠ label_ckd: dropped 42 column(s) that became all‑NaN
label_ckd: processed shape (2541, 391)
⚠ label_dyslipidemia: dropped 42 column(s) that became all‑NaN
label_dyslipidemia: processed shape (2399, 393)
⚠ label_cvd: dropped 42 column(s) that became all‑NaN
label_cvd: processed shape (2541, 391)
⚠ label_metabolic_syndrome: dropped 44 column(s) that became all‑NaN
label_metabolic_syndrome: processed shape (2192, 380)
⚠ label_liver_disease: dropped 42 column(s) that became all‑NaN
label_liver_disease: processed shape (2535, 392)

Done. Keys in processed_X: ['label_diabetes', 'label_hypertension', 'label_ckd', 'label_dyslipidemia', 'label_cvd', 'label_metabolic_syndrome', 'label_liver_disease']


## 6. Train/Test Split
Stratified split to preserve label distribution.

In [10]:
print("Keys in processed_X:", list(processed_X.keys()))

Keys in processed_X: ['label_diabetes', 'label_hypertension', 'label_ckd', 'label_dyslipidemia', 'label_cvd', 'label_metabolic_syndrome', 'label_liver_disease']


In [11]:
split_data = {}

for label_col in LABEL_COLS:
    X_proc = processed_X[label_col]
    y = label_sets[label_col]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_proc, y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y
    )
    
    split_data[label_col] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
    }
    
    print(f"{label_col}: train={X_train.shape[0]}, test={X_test.shape[0]}, "
          f"pos_train={y_train.sum()}, pos_test={y_test.sum()}")

label_diabetes: train=2016, test=504, pos_train=466, pos_test=117
label_hypertension: train=2032, test=509, pos_train=1180, pos_test=295
label_ckd: train=2032, test=509, pos_train=198, pos_test=49
label_dyslipidemia: train=1919, test=480, pos_train=1110, pos_test=278
label_cvd: train=2032, test=509, pos_train=256, pos_test=64
label_metabolic_syndrome: train=1753, test=439, pos_train=680, pos_test=170
label_liver_disease: train=2028, test=507, pos_train=289, pos_test=72


## 7. Save Processed Data
Save the preprocessed feature matrices, labels, and column information.

In [12]:
# Save the pipeline and data
import joblib

for label_col in LABEL_COLS:
    disease_name = label_col.replace("label_", "")
    
    # Save split arrays
    np.save(os.path.join(PROCESSED_DIR, f"{disease_name}_X_train.npy"), split_data[label_col]["X_train"])
    np.save(os.path.join(PROCESSED_DIR, f"{disease_name}_X_test.npy"), split_data[label_col]["X_test"])
    np.save(os.path.join(PROCESSED_DIR, f"{disease_name}_y_train.npy"), split_data[label_col]["y_train"])
    np.save(os.path.join(PROCESSED_DIR, f"{disease_name}_y_test.npy"), split_data[label_col]["y_test"])
    
    print(f"Saved {disease_name} arrays.")

# Also save the column names for the predictor set (for later explainability)
# For each disease, we saved the feature names (minus leakage) earlier – let's store them
feature_names = {}
for label_col in LABEL_COLS:
    exclude = LEAKAGE_MAP.get(label_col, [])
    cols = [c for c in PREDICTOR_COLS if c not in exclude]
    feature_names[label_col] = cols

with open(os.path.join(PROCESSED_DIR, "feature_names.json"), "w") as f:
    json.dump(feature_names, f, indent=2)

print("All processed data saved.")

Saved diabetes arrays.
Saved hypertension arrays.
Saved ckd arrays.
Saved dyslipidemia arrays.
Saved cvd arrays.
Saved metabolic_syndrome arrays.
Saved liver_disease arrays.
All processed data saved.


## Next Steps
Proceed to **04_model_training.ipynb** for model development, evaluation, and SHAP explainability.